# Évaluation des réponses générées

Évalue les citations, la couverture de mots-clés et prépare une revue humaine. Ce notebook appelle Ollama.


In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
cfg = load_config(ROOT / "config.yaml")
print(f"Projet : {ROOT}")


In [ ]:
import re, time, pandas as pd
from evaluation import load_golden_set, keyword_coverage
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline.from_config(ROOT / "config.yaml")
all_examples = load_golden_set(cfg.evaluation.expanded_golden_set_full_path)


In [ ]:
SPLIT = "development"
MAX_QUESTIONS = 12  # augmenter après ce test court
candidates = [item for item in all_examples if item.split == SPLIT and item.should_answer]

# Échantillon équilibré simple : un passage cyclique dans les catégories
selected, seen = [], set()
for item in candidates:
    if item.category not in seen:
        selected.append(item); seen.add(item.category)
for item in candidates:
    if item not in selected and len(selected) < MAX_QUESTIONS:
        selected.append(item)
selected = selected[:MAX_QUESTIONS]
print([item.example_id for item in selected])


In [ ]:
rows = []
for item in selected:
    started = time.perf_counter()
    response = pipeline.ask(item.question)
    mechanical_style = bool(re.search(
        r"(?im)^\s*(synthèse|réponse|liste .*attest|citation)\s*:",
        response.answer,
    ))
    rows.append({
        "id": item.example_id,
        "category": item.category,
        "question": item.question,
        "answer": response.answer,
        "reference_answer": item.reference_answer,
        "keyword_coverage": keyword_coverage(response.answer, item.answer_keywords),
        "answer_words": len(response.answer.split()),
        "answer_characters": len(response.answer),
        "answer_mode": pipeline.generator.last_prompt_stats.get("answer_mode"),
        "mechanical_style": mechanical_style,
        "citations": len(response.citations),
        "latency_seconds": time.perf_counter() - started,
        "human_correct": None,
        "human_faithful": None,
        "human_complete": None,
        "review_notes": "",
    })
generation_results = pd.DataFrame(rows)
display(generation_results[["id", "category", "answer_mode", "answer_words", "mechanical_style", "keyword_coverage", "citations", "latency_seconds"]])


In [ ]:
output_dir = cfg.evaluation.runs_full_path
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "generation_human_review.csv"
generation_results.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Grille de revue humaine : {output_path}")
